# What is ML and how do models learn?
**Topics:** Linear Regression · Logistic Regression · Loss Functions · Gradient Descent · SGD · Learning Rate


---
## 1. Linear Regression

### What it is
- Predicts a **continuous** output from input features
- Fits a straight line (or hyperplane) through data: `y = Xw + b`
- Minimizes the difference between predicted and actual values

### Key intuition
- Find the line that is "closest" to all data points
- Closeness measured by **squared errors** (penalizes large errors more)

### When to use
- Target variable is continuous (price, temperature, salary)
- Relationship between features and target is approximately linear
- Need an interpretable model — coefficients show feature impact

### When not to use
- Target is categorical → use logistic regression or classifiers
- Relationship is highly non-linear
- Many irrelevant features without regularization applied


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# ── From scratch ──
def linear_regression_scratch(X, y, lr=0.01, epochs=1000):
    m, n = X.shape
    w = np.zeros(n)
    b = 0
    for _ in range(epochs):
        y_pred = X @ w + b
        dw = (1/m) * X.T @ (y_pred - y)
        db = (1/m) * np.sum(y_pred - y)
        w -= lr * dw
        b -= lr * db
    return w, b

# ── scikit-learn ──
np.random.seed(42)
X = np.random.rand(100, 1) * 10
y = 3 * X.squeeze() + 7 + np.random.randn(100) * 2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Coefficient : {model.coef_[0]:.2f}")
print(f"Intercept   : {model.intercept_:.2f}")
print(f"MSE         : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R² Score    : {r2_score(y_test, y_pred):.2f}")


### Interview Q&A

**What does R² score mean?**
- Proportion of variance in y explained by the model
- R² = 1 → perfect fit, R² = 0 → model no better than predicting the mean

**What are the assumptions of linear regression?**
- Linearity, independence of errors, homoscedasticity (constant variance), normality of errors

**What happens if features are correlated (multicollinearity)?**
- Coefficients become unstable and hard to interpret → use Ridge regression (L2) to handle it

### Gotchas
- Sensitive to outliers — squared loss amplifies large errors
- Doesn't capture non-linear relationships without feature engineering
- Always check residual plots to validate assumptions


---
## 2. Logistic Regression

### What it is
- Predicts a **probability** that an input belongs to a class (0 or 1)
- Applies sigmoid function to linear output: `p = σ(Xw + b)`
- Despite the name — it's a **classification** algorithm

### Key intuition
- Squashes any real number into range (0, 1) using sigmoid
- Decision boundary: predict class 1 if p > 0.5, else class 0
- Learns weights that separate classes in feature space

### When to use
- Binary classification (spam/not spam, churn/no churn)
- Need probability outputs, not just class labels
- Need an interpretable baseline model

### When not to use
- Classes are not linearly separable → use SVM with kernel or tree-based models
- Multi-class with complex boundaries → use neural networks


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# ── From scratch ──
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_regression_scratch(X, y, lr=0.01, epochs=1000):
    m, n = X.shape
    w = np.zeros(n)
    b = 0
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        dw = (1/m) * X.T @ (p - y)
        db = (1/m) * np.sum(p - y)
        w -= lr * dw
        b -= lr * db
    return w, b

# ── scikit-learn ──
X, y = make_classification(n_samples=500, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print()
print(classification_report(y_test, y_pred))


### Interview Q&A

**Why is it called regression if it's a classifier?**
- It models the probability (a continuous value) using a regression approach, then thresholds it to produce a class label

**What loss function does logistic regression use?**
- Binary cross-entropy (log loss): penalizes confident wrong predictions heavily

**How do you extend it to multi-class?**
- One-vs-Rest (OvR): train one binary classifier per class
- Softmax (multinomial): single model outputting probabilities for all classes

### Gotchas
- Assumes linear decision boundary — fails on complex data
- Sensitive to class imbalance → use `class_weight='balanced'` or resample
- Features should be scaled for faster convergence


---
## 3. Loss Functions

### What it is
- Measures how wrong the model's predictions are
- Training = minimizing the loss function
- Choice of loss depends on the task

### Key intuition
- Loss is the "score" the model tries to lower
- Gradient descent follows the slope of the loss surface downhill

### Common loss functions

| Loss | Task | Formula (simplified) |
|---|---|---|
| MSE | Regression | mean((y - ŷ)²) |
| MAE | Regression (robust) | mean(|y - ŷ|) |
| Binary Cross-Entropy | Binary classification | -mean(y·log(p) + (1-y)·log(1-p)) |
| Categorical Cross-Entropy | Multi-class | -mean(Σ y·log(p)) |

### When to use
- MSE → standard regression, penalizes large errors heavily
- MAE → regression with outliers, more robust than MSE
- Cross-entropy → classification tasks, works with probability outputs


In [ ]:
import numpy as np

y_true = np.array([1.0, 2.0, 3.0, 4.0])
y_pred = np.array([1.2, 1.8, 3.5, 3.7])

# Mean Squared Error
mse = np.mean((y_true - y_pred) ** 2)

# Mean Absolute Error
mae = np.mean(np.abs(y_true - y_pred))

# Binary Cross-Entropy
y_true_cls = np.array([1, 0, 1, 1])
p_pred     = np.array([0.9, 0.1, 0.8, 0.6])
eps = 1e-8  # avoid log(0)
bce = -np.mean(y_true_cls * np.log(p_pred + eps) + (1 - y_true_cls) * np.log(1 - p_pred + eps))

print(f"MSE : {mse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"BCE : {bce:.4f}")


### Interview Q&A

**Why use MSE over MAE?**
- MSE is differentiable everywhere → easier to optimize with gradient descent
- MAE is more robust to outliers but has undefined gradient at 0

**Why cross-entropy for classification, not MSE?**
- MSE with sigmoid creates flat gradients (vanishing gradient problem)
- Cross-entropy gives strong gradients even when predictions are very wrong

### Gotchas
- Using MSE for classification → slow training, poor results
- Not scaling loss when batch sizes vary → use mean, not sum


---
## 4. Gradient Descent & Optimization

### What it is
- Iteratively adjusts weights to minimize the loss function
- Computes gradient of loss w.r.t. weights → steps in opposite direction

### Key intuition
- Blindfolded on a hilly landscape → feel the slope → step downhill
- Step size = **learning rate** — too large overshoots, too small is slow

### Types

| Type | Data per update | Pro | Con |
|---|---|---|---|
| Batch GD | Full dataset | Stable | Very slow on large data |
| SGD | 1 sample | Fast | Noisy |
| Mini-batch GD | n samples | Best of both | Needs batch size tuning |

### When to use
- Mini-batch GD → default for most ML/DL tasks
- Full batch GD → small datasets where stability matters
- SGD → online learning, streaming data


In [ ]:
import numpy as np

# ── Gradient Descent from scratch (Linear Regression) ──
def gradient_descent(X, y, lr=0.01, epochs=1000):
    m, n = X.shape
    w = np.zeros(n)
    b = 0
    losses = []
    for _ in range(epochs):
        y_pred = X @ w + b
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        dw = (1/m) * X.T @ (y_pred - y)
        db = (1/m) * np.sum(y_pred - y)
        w -= lr * dw
        b -= lr * db
    return w, b, losses

# ── Mini-batch GD ──
def mini_batch_gd(X, y, lr=0.01, epochs=100, batch_size=32):
    m, n = X.shape
    w = np.zeros(n)
    b = 0
    for _ in range(epochs):
        idx = np.random.permutation(m)
        X, y = X[idx], y[idx]
        for i in range(0, m, batch_size):
            Xb, yb = X[i:i+batch_size], y[i:i+batch_size]
            y_pred = Xb @ w + b
            dw = (1/len(yb)) * Xb.T @ (y_pred - yb)
            db = (1/len(yb)) * np.sum(y_pred - yb)
            w -= lr * dw
            b -= lr * db
    return w, b

# Demo
np.random.seed(42)
X = np.random.rand(200, 2)
y = 3*X[:,0] + 2*X[:,1] + np.random.randn(200)*0.1

w, b, losses = gradient_descent(X, y, lr=0.1, epochs=500)
print(f"Learned weights : {w}")
print(f"Learned bias    : {b:.4f}")
print(f"Final loss      : {losses[-1]:.6f}")


### Interview Q&A

**What happens if learning rate is too high?**
- Loss oscillates or diverges — overshoots the minimum

**SGD vs Adam?**
- SGD → fixed learning rate for all parameters
- Adam → adapts learning rate per parameter using gradient history → faster convergence

**Why does mini-batch generalize better than full batch?**
- Noise in updates acts as implicit regularization → escapes sharp minima → better generalization

### Gotchas
- Always normalize features first — elongated loss surface causes slow zigzag convergence
- Local minima rarely the issue in deep learning — saddle points are more common
- Learning rate is the most important hyperparameter to tune


---
## 5. Learning Rate

### What it is
- Controls the size of each step during gradient descent
- Single most important hyperparameter in training

### Key intuition
- Too high → overshoots minimum, loss diverges
- Too low → training takes forever, may get stuck
- Just right → smooth convergence to minimum

### When to use schedulers
- **Step decay** → reduce LR by factor every N epochs
- **Reduce on plateau** → reduce when validation loss stops improving
- **Cosine annealing** → smooth decay following cosine curve
- **Warmup** → start low, ramp up, then decay (common in transformers)


In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_regression(n_samples=500, n_features=5, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Fixed learning rate
model_fixed = SGDRegressor(learning_rate='constant', eta0=0.01, max_iter=1000)
model_fixed.fit(X_train, y_train)

# Adaptive learning rate (invscaling)
model_adaptive = SGDRegressor(learning_rate='invscaling', eta0=0.01, max_iter=1000)
model_adaptive.fit(X_train, y_train)

print(f"Fixed LR    R²: {model_fixed.score(X_test, y_test):.4f}")
print(f"Adaptive LR R²: {model_adaptive.score(X_test, y_test):.4f}")

# PyTorch-style scheduler (conceptual example)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
# After each epoch: scheduler.step(val_loss)


### Interview Q&A

**How do you choose an initial learning rate?**
- Start with 1e-3 for Adam, 1e-2 for SGD
- Use a learning rate range test — increase LR exponentially, plot loss, pick value just before loss spikes

**What is learning rate warmup?**
- Start with very small LR, gradually increase over first N steps
- Prevents large, destabilizing updates at the start of training
- Common in transformers (BERT, GPT)

### Gotchas
- Don't use same LR for all layers when fine-tuning — use lower LR for pretrained layers
- LR schedulers need to be stepped correctly — after optimizer step, not before
